In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

def calculate_daily_s_dot_average(target_dir):
    '''
    This function calculates the monthly average of the S_dot column from CSV files in the target directory.
    It reads each CSV file, extracts the S_dot column, and computes the monthly average.
    '''
    
    # get the list of CSV files in the target directory
    list_csv = os.listdir(target_dir)
    # filter the list to include only CSV files
    list_csv = [f for f in list_csv if f.endswith(".csv")]

    list_df = []
    for f in list_csv:
        # read the CSV file into a DataFrame
        df_tmp = pd.read_csv(os.path.join(target_dir, f), encoding="cp949", low_memory=False)
        df_tmp['측정시간'] = pd.to_datetime(df_tmp['측정시간'], format='%Y-%m-%d_%H:%M:%S')
        df_tmp['측정일'] = df_tmp['측정시간'].dt.date
        df_tmp['시간'] = df_tmp['측정시간'].dt.hour  # 시간 컬럼 추가 (0-23)
        target_columns = [c for c in df_tmp.columns if "평균" in c]
        target_columns.append("측정일")
        target_columns.append("시간")
        target_columns.append("시리얼")
        list_df.append(df_tmp.loc[:,target_columns].copy())
    
    df = pd.concat(list_df, ignore_index=True)
    target_columns = [c for c in df.columns if "평균" in c]
    df[target_columns] = df[target_columns].replace(to_replace=r'.*[A-Za-z].*', value=np.nan, regex=True).astype(float)
    # group by '측정일', '시간', and '시리얼' and calculate the mean of the target columns
    df = df.groupby(['측정일', '시간', '시리얼'], as_index=False).mean()
    return df

target_dir = "raw_data/s_dot_nature_2023"
# calculate the monthly average of the S_dot column
df = calculate_daily_s_dot_average(target_dir)

In [4]:
df_environment = df.loc[:, ['측정일','시간','시리얼','온도 평균(℃)', '습도 평균(%)', '소음 평균(dB)']]

# 컬럼 이름을 영어로 변경
column_mapping = {
    '측정일': 'date',
    '시간': 'hour',
    '시리얼': 'serial',
    '온도 평균(℃)': 'temperature_avg',
    '습도 평균(%)': 'humidity_avg',
    '소음 평균(dB)': 'noise_avg'
}

df_environment = df_environment.rename(columns=column_mapping)

# date와 hour를 datetime으로 합치기
df_environment['datetime'] = pd.to_datetime(
    df_environment['date'].astype(str) + ' ' + df_environment['hour'].astype(str) + ':00:00'
)

# 기존 date, hour 컬럼 제거 및 컬럼 순서 재정렬
df_environment = df_environment.drop(['date', 'hour'], axis=1)
df_environment = df_environment[['datetime', 'serial', 'temperature_avg', 'humidity_avg', 'noise_avg']]

# 데이터 타입 최적화 (Supabase 업로드용)
df_environment['serial'] = df_environment['serial'].astype('string')
float_cols = ['temperature_avg', 'humidity_avg', 'noise_avg']
for col in float_cols:
    df_environment[col] = df_environment[col].astype('float32')

df_environment

,datetime,serial,temperature_avg,humidity_avg,noise_avg
0,2023-01-01 01:00:00,OC3CL200010,-1.50,100.0,47.0
1,2023-01-01 01:00:00,OC3CL200011,NaN,NaN,58.0
2,2023-01-01 01:00:00,OC3CL200012,2.10,67.0,47.0
3,2023-01-01 01:00:00,OC3CL200013,1.40,70.0,64.0
4,2023-01-01 01:00:00,OC3CL200014,4.70,78.0,56.0
...,...,...,...,...,...
4232635,2023-12-24 12:00:00,V02Q2300003,-2.45,54.5,NaN
4232636,2023-12-24 12:00:00,V02Q2300004,-4.30,60.5,NaN
4232637,2023-12-24 12:00:00,V02Q2300005,-3.30,61.0,NaN
4232638,2023-12-24 12:00:00,V02Q2300006,-3.05,60.0,NaN


In [7]:
# datetime 기준으로 데이터를 절반씩 나누어 저장
df_sorted = df_environment.sort_values('datetime').reset_index(drop=True)

# 전체 데이터의 중간 지점 계산
midpoint = len(df_sorted) // 2

# 첫 번째 절반과 두 번째 절반으로 분할
df_first_half = df_sorted.iloc[:midpoint]
df_second_half = df_sorted.iloc[midpoint:]

# CSV 파일로 저장
df_first_half.to_csv('./sample/environment_2023_part1.csv', index=False)
df_second_half.to_csv('./sample/environment_2023_part2.csv', index=False)